In [1]:
import numpy as np
from scipy.sparse import load_npz, csr_matrix
from scipy.sparse.linalg import norm as sp_norm
from sklearn.metrics.pairwise import cosine_similarity
from collections import defaultdict
import time

In [2]:
# Load data
matrix = load_npz("../data/interaction_matrix.npz")
n_users, n_items = matrix.shape

In [3]:
# precomputations
item_popularity = np.array(matrix.sum(axis=0)).flatten()
user_activity = np.array(matrix.sum(axis=1)).flatten()
global_mean = matrix.sum() / matrix.nnz

In [4]:
def _mask_seen(scores, seen_indices):
    """Set scores of already-seen items to -inf so they are never recommended"""
    scores = scores.copy()
    scores[seen_indices] = -np.inf
    return scores

In [5]:
# Baseline 1: Random Recommendation
def recommend_random(user_id, k=10, train_mask=None, rng=None):
    """
    Recommend k items chosen uniformly at random from unseen items
    """
    if rng is None:
        rng = np.random.default_rng()
    seen = set(train_mask if train_mask is not None else matrix[user_id].indices)
    candidates = np.array([i for i in range(n_items) if i not in seen])
    if len(candidates) <= k:
        return candidates
    return rng.choice(candidates, size=k, replace=False)

In [6]:
# Baseline 2: Popularity-based recommendation
def recommend_popular(user_id, k= 10, train_mask=None):
    """
    Recommend the globally most popular items the user has not yet seen
    """
    seen = train_mask if train_mask is not None else matrix[user_id].indices
    scores = _mask_seen(item_popularity.astype(np.float64), seen)
    return np.argsort(scores)[::-1][:k]

In [7]:
# Baseline 3: User-based Collaborative Filtering
class UserBasedCF:
    """
    kNN user-based collaborative filtering using cosine similarity.
    """

    def __init__(self, n_neighbors=50):
        self.n_neighbors = n_neighbors
        self._sim = None

    def fit(self, mat):
        self._sim = cosine_similarity(mat)
        np.fill_diagonal(self._sim, 0.0) # exclude self-similarity
        self._mat = mat
        return self

    def recommend(self, user_id, k=10, train_mask=None):
        if self._sim is None:
            raise RuntimeError("Call .fit() before recommending")
        seen = train_mask if train_mask is not None else self._mat[user_id].indices

        # Get neighbors
        sim_row = self._sim[user_id] 
        top_neighbors = np.argsort(sim_row)[::-1][:self.n_neighbors]

        # Aggregate neighbor interactions weighted by similarity
        neighbor_sims  = sim_row[top_neighbors]
        neighbor_items = self._mat[top_neighbors]
        scores = np.array(neighbor_sims @ neighbor_items).flatten()

        scores = _mask_seen(scores, seen)
        return np.argsort(scores)[::-1][:k]

In [8]:
# Baseline 4: Item-based Collaborative Filtering
class ItemBasedCF:
    """
    kNN item-based collaborative filtering using cosine similarity.
    """

    def __init__(self, n_neighbors=50):
        self.n_neighbors = n_neighbors
        self._item_sim = None

    def fit(self, mat):
        self._item_sim = cosine_similarity(mat.T)
        np.fill_diagonal(self._item_sim, 0.0) # exclude self-similarity
        self._mat = mat
        return self

    def recommend(self, user_id, k=10, train_mask=None):
        if self._item_sim is None:
            raise RuntimeError("Call .fit() before recommending")
        seen = train_mask if train_mask is not None else self._mat[user_id].indices

        # use popularity if no interactions
        if len(seen) == 0:
            return recommend_popular(user_id, k, train_mask=seen)

        # get user interactions
        user_row = self._mat[user_id].toarray().flatten()
        weights = user_row[seen]

        # scores
        sim_sub = self._item_sim[:, seen]
        sorted_idx = np.argsort(sim_sub, axis=1)[:, ::-1]
        if self.n_neighbors < sorted_idx.shape[1]:
            topk_idx = sorted_idx[:, :self.n_neighbors]
        else:
            topk_idx = sorted_idx
        topk_sim = np.take_along_axis(sim_sub, topk_idx, axis=1)
        topk_weights = np.take_along_axis(weights[None, :], topk_idx, axis=1)
        numerator = (topk_sim * topk_weights).sum(axis=1)
        denom = np.abs(topk_sim).sum(axis=1) + 1e-8
        scores = numerator / denom

        # mask seen items
        scores = _mask_seen(scores, seen)
        return np.argsort(scores)[::-1][:k]

In [9]:
# Baseline 5: Bias-only model
class BiasOnlyModel:
    """
    Predicts rating/interaction strength as mean + user_bias + item_bias
    """

    def fit(self, mat):
        self._mat = mat
        n_u, n_i = mat.shape
        self.mu = mat.sum() / mat.nnz

        # User biases
        user_sums = np.array(mat.sum(axis=1)).flatten()
        user_nnz = np.diff(mat.indptr)                  # number non zero per row
        user_nnz = np.where(user_nnz == 0, 1, user_nnz) # avoid division by zero
        self.b_u = user_sums / user_nnz - self.mu

        # Item biases
        item_sums = np.array(mat.sum(axis=0)).flatten()
        item_nnz = np.array((mat != 0).sum(axis=0)).flatten()
        item_nnz = np.where(item_nnz == 0, 1, item_nnz)
        self.b_i = item_sums / item_nnz - self.mu

        return self

    def recommend(self, user_id, k=10, train_mask=None):
        seen = train_mask if train_mask is not None else self._mat[user_id].indices
        scores = self.mu + self.b_u[user_id] + self.b_i
        scores = _mask_seen(scores, seen)
        return np.argsort(scores)[::-1][:k]

In [13]:
def _dcg(relevances: np.ndarray) -> float:
    """Discounted Cumulative Gain"""
    return float(np.sum(relevances / np.log2(np.arange(2, len(relevances) + 2))))


def evaluate(recommend_func, mat, k=10, n_users=200, seed=123):
    """
    Leave one out evaluation
    """
    rng = np.random.default_rng(seed)
    users = rng.choice(mat.shape[0], size=min(n_users, mat.shape[0]), replace=False)
    
    hits, precision_sum, ndcg_sum, total = 0, 0.0, 0.0, 0
    
    for u in users:
        items = mat[u].indices
        # ensure enough items to do LOO eval
        if len(items) < 2:
            continue 

        # choose random interaction as test item
        test_item = rng.choice(items)
        train_mask = items[items != test_item]

        # get recommendations
        recs = recommend_func(u, k, train_mask)

        # compute metrics
        relevances = np.array([1 if r == test_item else 0 for r in recs])
        ideal = np.array([1] + [0] * (k - 1))
        hits += int(test_item in recs)         # hit rate @ k
        precision_sum += relevances.sum() / k  # precision @ k
        ndcg_sum += _dcg(relevances)           # discounted cumulative gain (position-aware)
        total += 1

    return {
        "hit_rate": hits / total,
        "precision": precision_sum / total,
        "ndcg": ndcg_sum / total,
        "n_evaluated": total,
    }

In [14]:
SEED = 123
K = 10
N_EVAL = 500
N_NEIGHBORS = 50
rng = np.random.default_rng(SEED)

# fit models that need fitting
user_cf = UserBasedCF(n_neighbors=N_NEIGHBORS).fit(matrix)
item_cf = ItemBasedCF(n_neighbors=N_NEIGHBORS).fit(matrix)
bias_model = BiasOnlyModel().fit(matrix)

# evaluate models
baselines = {
    "Random" : lambda u, k, msk: recommend_random(u, k, msk, rng),
    "Popularity" : recommend_popular,
    "UserCF" : user_cf.recommend,
    "ItemCF" : item_cf.recommend,
    "BiasOnly": bias_model.recommend
}

print(f"\nEvaluating baselines  (K={K}, n_users={N_EVAL}, seed={SEED})\n")
print(f"{'Baseline':<14}  {'Hit Rate':>9}  {'Precision':>9}  {'NDCG':>9}  {'Users':>6}  {'Time':>7}")
print("-" * 62)

results = {}
for name, fn in baselines.items():
    t0 = time.perf_counter()
    res = evaluate(fn, matrix, k=K, n_users=N_EVAL, seed=SEED)
    elapsed = time.perf_counter() - t0
    results[name] = res

    print(
        f"{name:<14}  "
        f"{res['hit_rate']:>9.4f}  "
        f"{res['precision']:>9.4f}  "
        f"{res['ndcg']:>9.4f}  "
        f"{res['n_evaluated']:>6}  "
        f"{elapsed:>6.1f}s"
    )


Evaluating baselines  (K=10, n_users=500, seed=123)

Baseline         Hit Rate  Precision       NDCG   Users     Time
--------------------------------------------------------------
Random             0.0040     0.0004     0.0014     500     0.5s
Popularity         0.0240     0.0024     0.0127     500     0.4s
UserCF             0.5120     0.0512     0.4135     500     0.3s
ItemCF             0.1260     0.0126     0.0993     500     5.1s
BiasOnly           0.0000     0.0000     0.0000     500     0.3s
